In [1]:
%pip install numpy pandas matplotlib seaborn scipy

python -m pip install numpy pandas matplotlib seaborn scipy jupyter
jupyter notebook


SyntaxError: invalid syntax (1101464918.py, line 3)

# Tarea 3 – Estadística descriptiva en Python (parte II)
**Universidad Industrial de Santander · Escuela de Matemáticas · Estadística I**
**Profesor:** Andrés Ríos-Gutiérrez  
**Integrantes:**  
- Cristhian Steven Ortiz Cardozo (2252138)  
- Daniel Stive Forero Mora (2250250)  

**Repositorio de GitHub:** _(pegar aquí el enlace de la rama)_

**Tema:** Tiempo de gestación (`T_GES`) y talla al nacer (`TALLA_NAC`) en los nacidos vivos de Colombia, 2024 (DANE, EEVV).

## 1. Introducción y contexto de los datos

**Objetivo.** Describir el comportamiento del tiempo de gestación y de la talla al nacer de los nacidos vivos en Colombia en 2024, y determinar si están asociadas, con qué grado y en qué sentido se puede hablar de dependencia.

**Origen de los datos.** Las Estadísticas Vitales (EEVV) de nacimientos del DANE se producen a partir de la información de los certificados de nacido vivo (<https://microdatos.dane.gov.co/index.php/catalog/878>). La base `BD-EEVV-Nacimientos-2024` tiene **453.901 nacimientos** y 40 variables (la variable `COD_DPTO` se usa como índice al cargarla). Cada fila es **un nacimiento**. Además de los certificados diligenciados por personal de salud en el aplicativo RUAF‑ND, el DANE integra registros recuperados del registro civil (RNEC) y nacimientos reportados con el formato de notificación étnica; la variable `TIPOFORMULARIO` identifica la fuente (DANE, 2026a).

**Variables de interés.** Ambas llegan **agrupadas en categorías ordenadas**, no en valores numéricos (DANE, 2026b):

| Variable | Código | Categoría |
|---|---|---|
| `T_GES` (semanas de gestación) | 1 / 2 / 3 / 4 / 5 | menos de 22 / 22 a 27 / 28 a 37 / 38 a 41 / 42 y más |
| | 6 / 9 | Ignorado / Sin información |
| `TALLA_NAC` (centímetros) | 1 / 2 / 3 / 4 / 5 / 6 | menos de 20 / 20–29 / 30–39 / 40–49 / 50–59 / 60 y más |
| | 9 | Sin información |

**Consecuencia metodológica.** Como son variables **categóricas ordinales**, no tiene sentido calcular media o desviación estándar de los códigos. Se describen con **tablas de frecuencias y proporciones, moda y categoría mediana**, y la relación entre ellas se estudia con la **tabla de contingencia**, las **probabilidades condicionales**, la **prueba chi cuadrado de independencia** y las **medidas de asociación basadas en su estadístico** (coeficiente phi, coeficiente de contingencia y V de Cramér).

**Hipótesis de contexto.** La OMS define el nacimiento prematuro como el ocurrido antes de completar 37 semanas y distingue extremadamente prematuro (< 28 semanas), muy prematuro (28 a < 32) y prematuro moderado a tardío (32 a < 37) (OMS, 2023). Los estándares internacionales INTERGROWTH‑21st muestran que la longitud mediana al nacer aumenta con la edad gestacional, de ≈ 44 cm a las 33 semanas a ≈ 51 cm a las 42 semanas (Villar et al., 2014). Por eso se espera una **asociación positiva y fuerte** entre `T_GES` y `TALLA_NAC`, y que sea la talla la que dependa biológicamente del tiempo de gestación.

## 2. Carga de la base de datos

In [ ]:
import io, os, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")          # estilo de los gráficos
plt.rcParams["figure.dpi"] = 120           # resolución en pantalla
SEED = 42                                  # semilla: resultados reproducibles
rng = np.random.default_rng(SEED)

ModuleNotFoundError: No module named 'numpy'

Se importan `pandas` (tablas), `numpy` (cálculo), `matplotlib` y `seaborn` (gráficos) y `scipy.stats` (chi cuadrado y asociación). La semilla garantiza que los remuestreos den siempre lo mismo.

In [ ]:
def leer_base(nombre="BD-EEVV-Nacimientos-2024"):
    # Colab: sube el .csv (o el .zip) con files.upload(); fuera de Colab: busca el archivo en la carpeta actual
    try:
        from google.colab import files                 # solo existe en Google Colab
        uploaded = files.upload()                      # abre el selector de archivos
        archivo = list(uploaded.keys())[0]             # nombre del archivo subido
        contenido = io.BytesIO(uploaded[archivo])      # bytes -> objeto tipo archivo
    except ImportError:
        archivo = next(f for f in os.listdir(".") if f.startswith(nombre) and f.endswith((".csv", ".zip")))
        contenido = open(archivo, "rb")
    if archivo.endswith(".zip"):                       # el DANE entrega el .csv dentro de un .zip
        with zipfile.ZipFile(contenido) as z:
            interno = [n for n in z.namelist() if n.endswith(".csv")][0]
            contenido = io.BytesIO(z.read(interno))
    return pd.read_csv(contenido, sep=",", index_col=0, encoding="latin-1")   # separador coma, 1.ª columna como índice

tabla = leer_base()
tabla.head(3)

- `files.upload()` abre el selector de archivos de Colab y devuelve `{nombre: bytes}`; `io.BytesIO` convierte esos bytes en un archivo que `pd.read_csv` puede leer; `zipfile` extrae el `.csv` si se subió el `.zip`.
- `sep=","` es el separador de columnas, `index_col=0` usa la primera columna (`COD_DPTO`) como índice y `encoding="latin-1"` evita errores con tildes y ñ.
- `head(3)` muestra las tres primeras filas para comprobar la lectura.

✏️ **Interpretación:** la tabla tiene una fila por nacimiento, con columnas del nacimiento (departamento, sitio del parto, sexo, peso, talla, tiempo de gestación, tipo de parto), de la madre y del padre, y la fuente del certificado (`TIPOFORMULARIO`). `COD_DPTO` quedó como índice (con valores repetidos, por eso más adelante se usa `reset_index`), de ahí que se vean 39 columnas y no 40. Las variables de interés aparecen como números enteros que son **códigos de categoría**, no medidas.

## 3. Exploración inicial

In [ ]:
print("Dimensiones (filas, columnas):", tabla.shape)       # 453.901 nacimientos x 39 columnas (+ el índice)
variables = ["T_GES", "TALLA_NAC"]
print(tabla[variables].dtypes, "\n")                        # tipo de dato de cada variable
print("Valores NaN:", tabla[variables].isna().sum().to_dict(), "\n")
for v in variables:                                         # frecuencia de cada código, ordenada
    print(f"--- {v} ---")
    print(tabla[v].value_counts().sort_index().to_string(), "\n")

✏️ **Interpretación:** la base tiene 453.901 nacimientos y 39 columnas (más el índice). Ambas variables están guardadas como enteros y no tienen `NaN`, pero esos enteros son **códigos**: `T_GES` toma los valores 1 a 6 y 9, y `TALLA_NAC` los valores 1 a 6 y 9. Los códigos 6 y 9 de `T_GES` (9.976 y 63 nacimientos) y el 9 de `TALLA_NAC` (10.216) significan «Ignorado» o «Sin información» según el diccionario: no son tiempos ni tallas y deben tratarse como faltantes. Además, las categorías extremas son muy raras: 20 nacimientos con menos de 22 semanas, 1 con talla menor de 20 cm y 9 de 60 cm o más. Calcular una media de los códigos (como si «3 = 28–37» fuera un número) no tendría sentido.

## 4. Recodificación y limpieza de datos

**Criterios (justificados con el diccionario del DANE y la literatura):**

1. **Etiquetas.** Se reemplazan los códigos por sus categorías (semanas y centímetros) y se declaran como **categorías ordenadas**, para que tablas y gráficos respeten el orden natural.
2. **Faltantes.** En `T_GES` los códigos **6 = «Ignorado»** y **9 = «Sin información»** y en `TALLA_NAC` el código **9 = «Sin información»** no son categorías de tiempo ni de talla (DANE, 2026b): se pasan a `NaN`. No se imputan, porque los faltantes se concentran en fuentes de certificado que no recogen estos datos (ver tabla por `TIPOFORMULARIO`; DANE, 2025).
3. **Casos completos.** Para el análisis conjunto solo se usan los nacimientos con ambas variables válidas.
4. **Valores atípicos.** No se eliminan: se revisa su plausibilidad biológica. Las tallas de menos de 30 cm solo son verosímiles en nacimientos extremadamente prematuros, y las de 60 cm o más son excepcionales frente a los estándares internacionales, donde el percentil 97 de la longitud a las 42 semanas es ≈ 54 cm (Villar et al., 2014). Son muy pocos casos y se conservan, pero se agrupan para el contraste de hipótesis (sección 7).

In [ ]:
# Etiquetas de las categorías (según el diccionario del DANE)
ETQ_GES = {1: "<22", 2: "22–27", 3: "28–37", 4: "38–41", 5: "≥42"}                       # semanas de gestación
ETQ_TALLA = {1: "<20", 2: "20–29", 3: "30–39", 4: "40–49", 5: "50–59", 6: "≥60"}        # centímetros
FALTANTES = {"T_GES": [6, 9], "TALLA_NAC": [9]}                                          # códigos "sin dato"
ORD_GES, ORD_TALLA = list(ETQ_GES.values()), list(ETQ_TALLA.values())                    # orden natural

# Sub-tabla con lo necesario; reset_index porque el índice original (COD_DPTO) tiene valores repetidos
df = tabla[["T_GES", "TALLA_NAC", "T_GES_AGRU_CIE", "TIPOFORMULARIO"]].reset_index(drop=True)
n0 = len(df)
ges_ok = ~df["T_GES"].isin(FALTANTES["T_GES"])               # True si T_GES tiene un dato válido
talla_ok = ~df["TALLA_NAC"].isin(FALTANTES["TALLA_NAC"])     # True si TALLA_NAC tiene un dato válido

# Categorías ordenadas; los códigos faltantes quedan como NaN
df["GES"] = pd.Categorical(df["T_GES"].where(ges_ok).map(ETQ_GES), categories=ORD_GES, ordered=True)
df["TALLA"] = pd.Categorical(df["TALLA_NAC"].where(talla_ok).map(ETQ_TALLA), categories=ORD_TALLA, ordered=True)

ges = df["GES"].dropna()                                      # T_GES válido
talla = df["TALLA"].dropna()                                  # TALLA_NAC válido
comp = df.dropna(subset=["GES", "TALLA"])                     # casos completos (ambas variables)

print(f"Registros iniciales: {n0:,}")
print(f"T_GES sin dato (6 o 9): {(~ges_ok).sum():,} ({(~ges_ok).mean():.1%})  |  válidos: {len(ges):,}")
print(f"TALLA_NAC sin dato (9): {(~talla_ok).sum():,} ({(~talla_ok).mean():.1%})  |  válidos: {len(talla):,}")
print(f"Casos completos: {len(comp):,} ({len(comp)/n0:.1%})")

In [ ]:
# ¿De dónde vienen los faltantes? Se cruzan con la fuente del certificado (TIPOFORMULARIO)
FUENTE = {1: "1 = Certificado RUAF-ND", 2: "2 = Notificación étnica", 3: "3 = Recuperación cruce RNEC"}
aux = pd.DataFrame({"fuente": df["TIPOFORMULARIO"].map(FUENTE), "sin_ges": ~ges_ok, "sin_talla": ~talla_ok})
faltan = aux.groupby("fuente").agg(registros=("fuente", "size"), sin_T_GES=("sin_ges", "sum"), sin_TALLA_NAC=("sin_talla", "sum"))
faltan["% sin T_GES"] = (100 * faltan["sin_T_GES"] / faltan["registros"]).round(1)
faltan["% sin TALLA_NAC"] = (100 * faltan["sin_TALLA_NAC"] / faltan["registros"]).round(1)
faltan

In [ ]:
# Plausibilidad biológica: tallas < 30 cm con gestación >= 28 semanas, y tallas >= 60 cm
muy_pequena = (df["TALLA_NAC"].isin([1, 2]) & df["T_GES"].isin([3, 4, 5])).sum()
muy_grande = (df["TALLA_NAC"] == 6).sum()
print(f"Talla < 30 cm con gestación de 28 semanas o más: {muy_pequena} casos")
print(f"Talla >= 60 cm: {muy_grande} casos")

✏️ **Interpretación:** tras pasar los códigos a `NaN` se pierden **10.039** valores de `T_GES` (2,2 %) y **10.216** de `TALLA_NAC` (2,3 %); quedan **442.981 casos completos (97,6 %)**. Los faltantes son **estructurales**: los formularios tipo 3 (recuperación por cruce con la RNEC, 8.303 casos) no tienen ninguno de los dos datos; los de notificación étnica (308) no tienen `T_GES` y el 39,6 % tampoco tiene talla; en cambio, en los certificados RUAF‑ND (tipo 1, 445.290) solo faltan el 0,3 % y el 0,4 %. Esto concuerda con la advertencia del DANE de que los registros recuperados de la RNEC traen sobre todo datos de identificación y de lugar (DANE, 2025). Como `T_GES` falta en todos los formularios tipo 2 y 3, **todos los casos completos provienen de certificados RUAF‑ND**: los resultados describen ese universo (98,1 % de los nacimientos). Por eso no se imputa. Sobre la plausibilidad biológica, hay 24 nacimientos con talla menor de 30 cm pero 28 semanas o más de gestación, y 9 con 60 cm o más (en total 0,007 % de los casos completos): son verosímilmente errores de digitación o casos excepcionales; se conservan y se agrupan en la sección 7.2, donde no afectan la conclusión.

## 5. T_GES: análisis univariado
**Sea G la variable aleatoria definida como el tiempo de gestación (en semanas) de un nacido vivo en Colombia en 2024**, medida en categorías ordenadas. Para variables categóricas se usan **proporciones** (frecuencia de cada valor dividida por el total), diagramas de barras y de sectores, la **moda** (categoría más frecuente) y la **categoría mediana** (primera categoría donde la proporción acumulada llega a 0,5).

In [ ]:
def tabla_frecuencias(s, orden):
    n = s.value_counts().reindex(orden, fill_value=0)                  # frecuencia absoluta de cada categoría
    tf = pd.DataFrame({"n": n, "proporción": n / n.sum()})             # proporción = n / total
    tf["%"] = (100 * tf["proporción"]).round(2)                        # porcentaje
    tf["% acumulado"] = (100 * tf["proporción"].cumsum()).round(2)     # porcentaje acumulado (válido por ser ordinal)
    return tf

def moda_mediana(tf):
    moda = tf["n"].idxmax()                                            # categoría con mayor frecuencia
    mediana = tf.index[(tf["proporción"].cumsum() >= 0.5).to_numpy().argmax()]   # primera con acumulado >= 0,5
    return moda, mediana

tf_ges = tabla_frecuencias(ges, ORD_GES)
moda_g, med_g = moda_mediana(tf_ges)

# Proporción pretérmino (< 37 semanas) con la agrupación de la CIE (T_GES_AGRU_CIE: 1, 2, 3 = <22, 22–27, 28–36)
cie = df.loc[~df["T_GES_AGRU_CIE"].isin([6, 9]), "T_GES_AGRU_CIE"]
print(f"Pretérmino (<37 sem.): {(cie <= 3).mean():.2%} | extremadamente prematuros (<28 sem.): {(cie <= 2).mean():.2%} | 42 semanas o más: {(cie == 5).mean():.2%}")
print(f"n válido = {tf_ges['n'].sum():,} | moda = {moda_g} semanas | categoría mediana = {med_g} semanas")
tf_ges

In [ ]:
def barras(tf, xlabel, color, archivo):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))                     # dos paneles: escala lineal y logarítmica
    for a, log in zip(ax, [False, True]):
        a.bar(tf.index.astype(str), tf["n"], color=color, edgecolor="black")      # barras de frecuencia
        for i, (n_, p_) in enumerate(zip(tf["n"], tf["%"])):                      # etiqueta con el porcentaje
            a.text(i, max(n_, 1) * (1.2 if log else 1.01), (f"{p_:.2f} %" if p_ >= 0.01 else "<0,01 %"), ha="center", va="bottom", fontsize=9)
        if log:
            a.set_yscale("log"); a.set_ylim(0.8, tf["n"].max() * 8)             # escala log para ver categorías raras
        else:
            a.set_ylim(0, tf["n"].max() * 1.12)
        a.set(xlabel=xlabel, ylabel="Frecuencia" + (" (escala logarítmica)" if log else ""))
    ax[0].set_title("Diagrama de barras"); ax[1].set_title("Diagrama de barras (escala logarítmica)")
    plt.tight_layout(); plt.savefig(archivo, dpi=200); plt.show()

def sectores(tf, titulo, colores, archivo):
    plt.figure(figsize=(6.5, 5))
    vistos = tf[tf["n"] > 0]                                                    # solo categorías con datos
    plt.pie(vistos["n"], colors=colores[:len(vistos)], startangle=90, counterclock=False,
            autopct=lambda p: f"{p:.1f} %" if p >= 2 else "", wedgeprops={"edgecolor": "white"})
    plt.legend([f"{c}: {p:.2f} %" for c, p in zip(vistos.index, vistos["%"])], title="Categoría",
               loc="center left", bbox_to_anchor=(1, 0.5))                       # leyenda con el % de cada categoría
    plt.title(titulo); plt.tight_layout(); plt.savefig(archivo, dpi=200); plt.show()

barras(tf_ges, "Tiempo de gestación (semanas)", "skyblue", "fig_t_ges_barras.png")

In [ ]:
sectores(tf_ges, "Tiempo de gestación (T_GES): diagrama de sectores", sns.color_palette("Blues", 5), "fig_t_ges_sectores.png")

✏️ **Interpretación:** con **443.862** nacimientos válidos, el tiempo de gestación se concentra en una sola categoría: **38 a 41 semanas** es la **moda y la categoría mediana**, con el **72,17 %** de los nacimientos (320.319); le sigue **28 a 37 semanas** con el 27,36 % (121.446). Las demás son raras: 22 a 27 semanas el 0,39 % (1.712), 42 o más el 0,08 % (365) y menos de 22 semanas solo 20 nacimientos (< 0,01 %). Dos categorías reúnen el 99,5 % de los datos. La distribución es **asimétrica hacia las gestaciones cortas** (la cola larga está del lado de menos semanas): el porcentaje acumulado ya llega al 99,92 % en 38–41 semanas. Con la clasificación de la CIE que ofrece el propio DANE (`T_GES_AGRU_CIE`, que separa 28–36 de 37–41 semanas), los nacimientos **pretérmino (< 37 semanas) son el 11,12 %** y los extremadamente prematuros (< 28 semanas) el 0,39 %: del orden de 1 de cada 9 nacimientos, comparable con la estimación de la OMS de más de 1 de cada 10 nacidos antes de las 37 semanas en el mundo (OMS, 2023). Con la agrupación original no se pueden aislar los prematuros, porque «28 a 37» mezcla pretérmino con los nacidos en la semana 37. En el diagrama en escala logarítmica se ven las categorías raras, invisibles en la escala lineal; el de sectores confirma el predominio de dos categorías.

## 6. TALLA_NAC: análisis univariado
**Sea L la variable aleatoria definida como la talla (en centímetros) al nacer de un nacido vivo en Colombia en 2024**, medida en categorías ordenadas de 10 cm. Se aplica el mismo procedimiento que para G.

In [ ]:
tf_talla = tabla_frecuencias(talla, ORD_TALLA)
moda_t, med_t = moda_mediana(tf_talla)
print(f"n válido = {tf_talla['n'].sum():,} | moda = {moda_t} cm | categoría mediana = {med_t} cm")
tf_talla

In [ ]:
barras(tf_talla, "Talla al nacer (cm)", "lightgreen", "fig_talla_barras.png")
sectores(tf_talla, "Talla al nacer (TALLA_NAC): diagrama de sectores", sns.color_palette("Greens", 6), "fig_talla_sectores.png")

✏️ **Interpretación:** con **443.685** nacimientos válidos, la talla se concentra en dos categorías: **50 a 59 cm** (moda, 50,99 %; 226.231) y **40 a 49 cm** (47,94 %; 212.707), que juntas reúnen el 98,9 %. Siguen 30 a 39 cm (1,02 %; 4.509), 20 a 29 cm (0,05 %; 228) y casos aislados con menos de 20 cm (1) y con 60 cm o más (9). La **categoría mediana es 50–59 cm**, pero por muy poco: el acumulado en 40–49 cm es 49,01 %, de modo que la talla mediana está cerca de los 50 cm, el límite entre las dos categorías principales. Esto es consistente con los estándares INTERGROWTH‑21st, donde la longitud mediana de un recién nacido a término está entre ≈ 48 y 51 cm (Villar et al., 2014). La distribución es asimétrica hacia las tallas bajas (cola en 30–39 y 20–29 cm, que corresponden sobre todo a prematuros). Como las categorías miden 10 cm, **no se puede describir la forma de la distribución dentro de cada categoría**, y el corte de 50 cm cae en el centro de la distribución de los nacidos a término, lo que limita la información de la variable.

## 7. Relación entre T_GES y TALLA_NAC

### 7.1 Tabla de contingencia
La **tabla de contingencia** cuenta cuántos nacimientos hay en cada combinación de categorías de las dos variables (solo casos completos). Con 5 × 6 = 30 celdas, muchas quedan con muy pocos nacimientos.

In [ ]:
ct_orig = pd.crosstab(comp["GES"], comp["TALLA"]).reindex(index=ORD_GES, columns=ORD_TALLA, fill_value=0)   # frecuencias observadas
ct_tot = ct_orig.copy()
ct_tot.loc["Total"] = ct_orig.sum()               # total por columna (marginal de TALLA_NAC)
ct_tot["Total"] = ct_tot.sum(axis=1)              # total por fila (marginal de T_GES)
ct_tot

### 7.2 Supuesto de la prueba chi cuadrado y agrupación de categorías
La aproximación chi cuadrado exige **frecuencias esperadas suficientes**: la regla de Cochran (1954) pide que **ninguna esperada sea menor que 1 y que no más del 20 % sea menor que 5**. La frecuencia esperada de cada celda es (total de la fila × total de la columna) / n, es decir, lo que habría si las variables fueran independientes.

In [ ]:
chi2_o, p_o, gl_o, esp_o = stats.chi2_contingency(ct_orig)     # esp_o = frecuencias esperadas bajo independencia
menores5 = (esp_o < 5).sum()                                   # celdas con esperada < 5
print(f"Tabla original {ct_orig.shape[0]}x{ct_orig.shape[1]}: {menores5} de {esp_o.size} celdas ({menores5/esp_o.size:.1%}) con esperada < 5; mínima = {esp_o.min():.2g}")
print("¿Cumple Cochran (<= 20 % con esperada < 5 y ninguna < 1)?", menores5/esp_o.size <= 0.20 and esp_o.min() >= 1)

La tabla original **no cumple** el supuesto, por lo que no se usa para el contraste. Se agrupan las categorías con muy pocos casos con la categoría vecina, siguiendo el orden natural:

- `T_GES`: «< 22» y «22–27» → **«< 28»** (extremadamente prematuros, según la OMS). Quedan 4 categorías: < 28, 28–37, 38–41 y ≥ 42 semanas.
- `TALLA_NAC`: «< 20», «20–29» y «30–39» → **«< 40»**; «50–59» y «≥ 60» → **«≥ 50»**. Quedan 3 categorías: < 40, 40–49 y ≥ 50 cm.

In [ ]:
AGR_GES = {"<22": "<28", "22–27": "<28", "28–37": "28–37", "38–41": "38–41", "≥42": "≥42"}
AGR_TALLA = {"<20": "<40", "20–29": "<40", "30–39": "<40", "40–49": "40–49", "50–59": "≥50", "≥60": "≥50"}
ORD_GES_G, ORD_TALLA_G = ["<28", "28–37", "38–41", "≥42"], ["<40", "40–49", "≥50"]

comp = comp.assign(                                            # variables agrupadas (siguen siendo ordinales)
    GES_G=pd.Categorical(comp["GES"].astype(object).map(AGR_GES), categories=ORD_GES_G, ordered=True),
    TALLA_G=pd.Categorical(comp["TALLA"].astype(object).map(AGR_TALLA), categories=ORD_TALLA_G, ordered=True))

ct = pd.crosstab(comp["GES_G"], comp["TALLA_G"]).reindex(index=ORD_GES_G, columns=ORD_TALLA_G, fill_value=0)   # tabla agrupada 4x3
n = int(ct.values.sum())                                       # total de casos completos
ct_t = ct.copy(); ct_t.loc["Total"] = ct.sum(); ct_t["Total"] = ct_t.sum(axis=1)
print(f"n = {n:,}")
ct_t

In [ ]:
chi2, p, gl, esp_arr = stats.chi2_contingency(ct, correction=False)    # chi cuadrado de independencia (sin corrección: tabla mayor que 2x2)
esp = pd.DataFrame(esp_arr, index=ct.index, columns=ct.columns)        # frecuencias esperadas bajo independencia
menores5 = (esp_arr < 5).sum()
print(f"Tabla agrupada {ct.shape[0]}x{ct.shape[1]}: {menores5} de {esp_arr.size} celdas ({menores5/esp_arr.size:.1%}) con esperada < 5; mínima = {esp_arr.min():.2f}")
print("¿Cumple Cochran?", menores5/esp_arr.size <= 0.20 and esp_arr.min() >= 1)
esp.round(1)

### 7.3 Proporciones y probabilidades condicionales
Con `n` casos completos, cada celda tiene una **proporción conjunta** P(G = i, L = j) = n_ij / n. Las **probabilidades condicionales** se obtienen dividiendo cada fila (o columna) por su total:

- P(L = j | G = i) = n_ij / n_i·: distribución de la talla **dado** el tiempo de gestación (proporciones por fila).
- P(G = i | L = j) = n_ij / n_·j: distribución del tiempo de gestación **dada** la talla (proporciones por columna).

Si las variables fueran **independientes**, las probabilidades condicionales serían iguales a las marginales en todas las filas (o columnas).

In [ ]:
conj = ct / n                                              # proporciones conjuntas P(G=i, L=j)
marg_G = ct.sum(axis=1) / n                                # marginal de T_GES: P(G=i)
marg_L = ct.sum(axis=0) / n                                # marginal de TALLA_NAC: P(L=j)
cond_L_dado_G = ct.div(ct.sum(axis=1), axis=0)             # P(L=j | G=i): cada fila suma 1
cond_G_dado_L = ct.div(ct.sum(axis=0), axis=1)             # P(G=i | L=j): cada columna suma 1

print("Proporciones conjuntas P(G, L):");            print(conj.round(4).to_string(), "\n")
print("Marginal de T_GES, P(G):");                    print(marg_G.round(4).to_string(), "\n")
print("Marginal de TALLA_NAC, P(L):");                print(marg_L.round(4).to_string(), "\n")
print("Condicional P(TALLA | T_GES)  [filas suman 1]:");   print(cond_L_dado_G.round(4).to_string(), "\n")
print("Condicional P(T_GES | TALLA)  [columnas suman 1]:"); print(cond_G_dado_L.round(4).to_string())

✏️ **Interpretación:** las **proporciones conjuntas** más altas son 38–41 semanas con 50 cm o más (43,86 %), 38–41 con 40–49 cm (28,28 %) y 28–37 con 40–49 cm (19,60 %). Las **marginales** son P(talla ≥ 50) = 51,0 % y P(talla < 40) = 1,07 %. Si las variables fueran independientes, la probabilidad de medir 50 cm o más sería ≈ 51 % en **todos** los grupos de gestación; las **condicionales** muestran lo contrario: P(≥ 50 | < 28 semanas) = 0 %, P(≥ 50 | 28–37) = 25,9 %, P(≥ 50 | 38–41) = 60,8 % y P(≥ 50 | ≥ 42) = 74,7 %. A la inversa, P(< 40 cm | < 28 semanas) = 96,4 %, frente a P(< 40 | 28–37) = 2,5 % y casi 0 % en 38–41 semanas. Es decir, **la distribución de la talla cambia con el tiempo de gestación**: a más semanas, mayor probabilidad de talla alta. Las condicionales en el otro sentido apuntan igual: entre los nacidos con 50 cm o más, el 86,0 % tiene 38–41 semanas, frente al 59,0 % entre los de 40–49 cm; entre los de menos de 40 cm, el 35,2 % es de < 28 semanas y el 64,3 % de 28–37. Ojo: P(A | B) y P(B | A) responden preguntas distintas y no son iguales.

### 7.4 Gráficos de la tabla de contingencia

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
# (1) Mapa de calor de frecuencias (color en escala log10 para ver las celdas pequeñas; el número es la frecuencia)
sns.heatmap(np.log10(ct + 1), annot=ct, fmt="d", cmap="Blues", cbar_kws={"label": "log10(frecuencia + 1)"}, ax=ax[0])
ax[0].set(title="Tabla de contingencia: frecuencias", xlabel="Talla al nacer (cm)", ylabel="Tiempo de gestación (semanas)"); ax[0].tick_params(axis="y", rotation=0)
# (2) Mapa de calor de P(TALLA | T_GES) en porcentaje
sns.heatmap(100 * cond_L_dado_G, annot=True, fmt=".1f", cmap="Greens", vmin=0, vmax=100, cbar_kws={"label": "% dentro de cada fila"}, ax=ax[1])
ax[1].set(title="P(TALLA | T_GES) en %", xlabel="Talla al nacer (cm)", ylabel="Tiempo de gestación (semanas)"); ax[1].tick_params(axis="y", rotation=0)
plt.tight_layout(); plt.savefig("fig_contingencia_mapas.png", dpi=200); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
pal3 = sns.color_palette("YlGnBu", 3); pal4 = sns.color_palette("YlOrRd", 4)
# (1) Barras apiladas al 100 %: distribución de la talla dentro de cada categoría de gestación
cond_L_dado_G.plot(kind="bar", stacked=True, color=pal3, edgecolor="black", rot=0, ax=ax[0])
for i, tot in enumerate(ct.sum(axis=1)):                         # tamaño de cada grupo encima de su barra
    ax[0].text(i, 1.02, f"n = {tot:,}", ha="center", fontsize=9)
ax[0].set(title="Talla al nacer según tiempo de gestación", xlabel="Tiempo de gestación (semanas)", ylabel="Proporción", ylim=(0, 1.1))
ax[0].legend(title="Talla (cm)", loc="center left", bbox_to_anchor=(1.01, 0.5))
# (2) Barras apiladas al 100 %: distribución de la gestación dentro de cada categoría de talla
cond_G_dado_L.T.plot(kind="bar", stacked=True, color=pal4, edgecolor="black", rot=0, ax=ax[1])
for i, tot in enumerate(ct.sum(axis=0)):
    ax[1].text(i, 1.02, f"n = {tot:,}", ha="center", fontsize=9)
ax[1].set(title="Tiempo de gestación según talla al nacer", xlabel="Talla al nacer (cm)", ylabel="Proporción", ylim=(0, 1.1))
ax[1].legend(title="Gestación (sem.)", loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(); plt.savefig("fig_contingencia_barras.png", dpi=200); plt.show()

✏️ **Interpretación:** el mapa de calor de frecuencias (colores en escala logarítmica) muestra que los datos se concentran en las filas de 28 a 41 semanas y las columnas de 40 a 59 cm, con una «diagonal» clara: las gestaciones más cortas se asocian a tallas más bajas. El mapa de P(talla | gestación) lo hace más evidente: la fila de < 28 semanas es 96,4 % «< 40 cm»; la de 28–37 semanas se reparte entre 40–49 cm (71,6 %) y ≥ 50 cm (25,9 %); y las de 38–41 y ≥ 42 semanas pasan a estar dominadas por «≥ 50 cm» (60,8 % y 74,7 %). En las barras apiladas al 100 % del primer panel, el bloque oscuro (≥ 50 cm) crece de izquierda a derecha; en el segundo panel, los de ≥ 50 cm son casi todos de 38–41 semanas. Si las variables fueran independientes, todas las barras de cada panel tendrían la misma composición; aquí son claramente distintas. Las barras de < 28 y de ≥ 42 semanas se basan en muy pocos casos (1.724 y 364), por lo que se anotó el tamaño de cada grupo.

### 7.5 Prueba chi cuadrado de independencia
- **H₀:** `T_GES` y `TALLA_NAC` son **independientes** (la distribución de la talla es la misma en todas las categorías de gestación).
- **H₁:** `T_GES` y `TALLA_NAC` **no son independientes** (están asociadas).
- **Estadístico:** $\chi^2=\sum_{i,j}\dfrac{(O_{ij}-E_{ij})^2}{E_{ij}}$, con $E_{ij}=\dfrac{n_{i\cdot}\,n_{\cdot j}}{n}$; bajo H₀ tiene aproximadamente distribución chi cuadrado con $(r-1)(c-1)$ grados de libertad.
- **Regla:** si el valor p < 0,05 se rechaza H₀ con una confianza del 95 %.

In [ ]:
chi2_manual = (((ct - esp) ** 2) / esp).to_numpy().sum()          # estadístico calculado a mano (control)
critico = stats.chi2.ppf(0.95, gl)                                 # valor crítico al 5 %
print(f"chi2 (scipy)  = {chi2:,.2f}")
print(f"chi2 (manual) = {chi2_manual:,.2f}")
print(f"grados de libertad = ({ct.shape[0]}-1)*({ct.shape[1]}-1) = {gl}")
print(f"valor crítico chi2(0,95; {gl}) = {critico:.3f}")
print(f"valor p = {p:.3e}   (0 significa menor que el mínimo representable por el computador)")
print("Decisión:", "se rechaza H0" if p < 0.05 else "no se rechaza H0")

In [ ]:
# Residuos estandarizados ajustados: ¿qué celdas se alejan más de la independencia? (|valor| > 2 es llamativo)
obs = ct.to_numpy(); fp = obs.sum(1) / n; cp = obs.sum(0) / n
res_aj = (obs - esp_arr) / np.sqrt(esp_arr * np.outer(1 - fp, 1 - cp))
res_aj = pd.DataFrame(res_aj, index=ct.index, columns=ct.columns)
plt.figure(figsize=(6.5, 4.2))
sns.heatmap(res_aj, annot=True, fmt=".0f", cmap="coolwarm", center=0, cbar_kws={"label": "residuo estandarizado ajustado"})
plt.title("Residuos estandarizados ajustados"); plt.xlabel("Talla al nacer (cm)"); plt.ylabel("Tiempo de gestación (semanas)"); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig("fig_residuos.png", dpi=200); plt.show()

✏️ **Interpretación:** **Prueba:** χ² = 194.518,9 con 6 grados de libertad, (4 − 1)(3 − 1), y valor crítico χ²₀,₉₅(6) = 12,59. El valor p es menor que el mínimo representable (≈ 0), por lo que **se rechaza H₀ con una confianza del 95 %**: `T_GES` y `TALLA_NAC` **no son independientes**. El supuesto de Cochran se cumple en la tabla agrupada (1 de 12 celdas, 8,3 %, con esperada menor que 5; la mínima es 3,88, en «≥ 42 semanas × < 40 cm»), mientras que la tabla original de 5 × 6 no lo cumplía (14 de 30 celdas). **Residuos estandarizados ajustados:** casi todas las celdas superan |2| y varias son enormes: «< 28 semanas × < 40 cm» (+386, muchísimos más nacimientos de los esperados bajo independencia), «28–37 × 40–49» (+194) y «38–41 × ≥ 50» (+209); y negativos en «28–37 × ≥ 50» (−205), «38–41 × 40–49» (−187) y «38–41 × < 40» (−111). Es decir, hay **más** nacimientos de lo esperado en la «diagonal» (menos semanas y menos talla; más semanas y más talla) y **menos** fuera de ella. La única celda en el límite es «≥ 42 × < 40» (−2,0), con esperada de solo 3,9. **Advertencia:** con n = 442.981 el chi cuadrado rechazaría la independencia incluso con diferencias mínimas; el rechazo muestra que hay asociación, no su fuerza, que se mide a continuación.

### 7.6 Medidas de asociación basadas en el estadístico chi cuadrado
El estadístico chi cuadrado crece con *n*, por lo que no mide **qué tan fuerte** es la asociación: con cientos de miles de datos casi cualquier diferencia es «significativa». Las siguientes medidas lo **normalizan** por el tamaño de muestra y quedan entre 0 (independencia) y un máximo:

| Medida | Fórmula | Rango |
|---|---|---|
| Coeficiente **φ** (generalizado, $w$ de Cohen) | $\sqrt{\chi^2/n}$ | 0 a $\sqrt{\min(r,c)-1}$ |
| Coeficiente de **contingencia C** de Pearson | $\sqrt{\chi^2/(\chi^2+n)}$ | 0 a $\sqrt{(k-1)/k}$, con $k=\min(r,c)$ |
| **C corregido** | $C/C_{\max}$ | 0 a 1 |
| **V de Cramér** | $\sqrt{\chi^2/\big(n\,(\min(r,c)-1)\big)}$ | 0 a 1 |
| **T de Tschuprow** | $\sqrt{\chi^2/\big(n\sqrt{(r-1)(c-1)}\big)}$ | 0 a 1 |

Para la V de Cramér se usa la escala de Cohen (1988) ajustada por los grados de libertad $df^*=\min(r,c)-1$: efecto pequeño ≈ $0{,}1/\sqrt{df^*}$, mediano ≈ $0{,}3/\sqrt{df^*}$ y grande ≈ $0{,}5/\sqrt{df^*}$.

In [ ]:
r_, c_ = ct.shape; k_ = min(r_, c_); dfs = k_ - 1                    # filas, columnas, k = min(r,c), df* = k-1
phi = np.sqrt(chi2 / n)                                               # phi generalizado (w de Cohen)
C = np.sqrt(chi2 / (chi2 + n))                                        # coeficiente de contingencia
Cmax = np.sqrt((k_ - 1) / k_); C_corr = C / Cmax                      # máximo posible y C corregido
V = np.sqrt(chi2 / (n * dfs))                                         # V de Cramér
T = np.sqrt(chi2 / (n * np.sqrt((r_ - 1) * (c_ - 1))))                # T de Tschuprow

# V de Cramér corregida por sesgo (Bergsma, 2013)
phi2c = max(0, chi2 / n - (r_ - 1) * (c_ - 1) / (n - 1))
rc = r_ - (r_ - 1) ** 2 / (n - 1); cc = c_ - (c_ - 1) ** 2 / (n - 1)
V_corr = np.sqrt(phi2c / min(rc - 1, cc - 1))

# IC95 % bootstrap de V: se remuestrea la tabla completa (n pares) con distribución multinomial
prob = (obs / n).ravel(); Vb = []
for _ in range(500):
    tab_b = rng.multinomial(n, prob).reshape(obs.shape)               # tabla remuestreada
    Vb.append(np.sqrt(stats.chi2_contingency(tab_b, correction=False)[0] / (n * dfs)))
V_lo, V_hi = np.percentile(Vb, [2.5, 97.5])

def nivel_V(v, dfs):
    # Escala de Cohen (1988) para V según df* = min(r,c)-1
    a = 0.1 / np.sqrt(dfs), 0.3 / np.sqrt(dfs), 0.5 / np.sqrt(dfs)
    return "despreciable" if v < a[0] else "pequeño" if v < a[1] else "mediano" if v < a[2] else "grande"

medidas = pd.DataFrame({"valor": [phi, C, C_corr, V, V_corr, T]},
                       index=["phi (w de Cohen)", "Contingencia C", "C corregido (C/Cmax)", "V de Cramér", "V de Cramér (corr. sesgo)", "T de Tschuprow"])
print(f"C máximo posible = {Cmax:.3f} | IC95 % bootstrap de V = [{V_lo:.3f}, {V_hi:.3f}] | nivel de V (Cohen, df*={dfs}): {nivel_V(V, dfs)}")
medidas.round(4)

✏️ **Interpretación:** las medidas basadas en χ² normalizan por el tamaño de muestra: **φ = 0,663**, **C = 0,552** (máximo posible 0,816, de modo que **C corregido = 0,677**), **V de Cramér = 0,469** (IC 95 % bootstrap [0,461; 0,475]; corregida por sesgo, 0,469) y **T de Tschuprow = 0,423**. Todas indican una asociación **fuerte**: con la escala de Cohen ajustada por df* = 2 (pequeño ≈ 0,07; mediano ≈ 0,21; grande ≈ 0,35), V = 0,47 es un efecto **grande**. El intervalo de V es muy estrecho (± 0,007) y la corrección por sesgo no cambia nada, porque n es enorme. T es menor que V porque la tabla no es cuadrada (4 × 3), y C no llegaría a 1 ni con asociación perfecta (su máximo es 0,816). **En palabras:** el tiempo de gestación y la talla están fuertemente asociados; las diferencias en la distribución de la talla entre grupos de gestación son grandes, no solo estadísticamente detectables.

### 7.7 Comparación con otras metodologías
El chi cuadrado **ignora el orden** de las categorías y es **simétrico**. Como ambas variables son ordinales y se pregunta «qué tan dependiente es una de la otra», se complementa con:

- **Razón de verosimilitud (prueba G):** alternativa asintóticamente equivalente al chi cuadrado, basada en $G=2\sum O\ln(O/E)$.
- **Medidas ordinales:** gamma de Goodman–Kruskal y tau‑b de Kendall (miden concordancia entre pares ordenados) y rho de Spearman.
- **Medidas asimétricas** (cuánto mejora la predicción de una variable al conocer la otra): lambda de Goodman–Kruskal (reducción proporcional del error de predicción) y coeficiente de incertidumbre de Theil, $U(Y|X)=\big(H(Y)-H(Y|X)\big)/H(Y)$ (reducción proporcional de entropía).

In [ ]:
# Prueba G (razón de verosimilitud)
G, pG, _, _ = stats.chi2_contingency(ct, correction=False, lambda_="log-likelihood")

# Medidas ordinales sobre las categorías agrupadas, codificadas 1,2,3,...
cod_g = comp["GES_G"].cat.codes.to_numpy() + 1                       # código ordinal de T_GES agrupado
cod_t = comp["TALLA_G"].cat.codes.to_numpy() + 1                     # código ordinal de TALLA_NAC agrupado
tau_b = stats.kendalltau(cod_g, cod_t, variant="b")[0]               # tau-b de Kendall (corrige empates)
rho_s = stats.spearmanr(cod_g, cod_t)[0]                             # rho de Spearman

def gamma_gk(tab):
    # gamma = (C - D) / (C + D), con C pares concordantes y D discordantes (recorre la tabla)
    tab = np.asarray(tab); C_ = D_ = 0
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]):
            C_ += tab[i, j] * tab[i + 1:, j + 1:].sum()
            D_ += tab[i, j] * tab[i + 1:, :j].sum()
    return (C_ - D_) / (C_ + D_)

def lambda_gk(tab):
    # lambda de Goodman-Kruskal para predecir las COLUMNAS conociendo la FILA
    tab = np.asarray(tab); nt = tab.sum()
    return (tab.max(axis=1).sum() - tab.sum(axis=0).max()) / (nt - tab.sum(axis=0).max())

def entropia(p):
    p = p[p > 0]; return -(p * np.log(p)).sum()

def theil_u(tab):
    # U(columnas | filas) = (H(col) - H(col | fila)) / H(col)
    p = np.asarray(tab) / np.sum(tab); pf = p.sum(axis=1); pc = p.sum(axis=0)
    H_c = entropia(pc)
    H_c_f = sum(pf[i] * entropia(p[i] / pf[i]) for i in range(len(pf)) if pf[i] > 0)
    return (H_c - H_c_f) / H_c

otros = pd.DataFrame({
    "valor": [G, gamma_gk(obs), tau_b, rho_s, lambda_gk(obs), lambda_gk(obs.T), theil_u(obs), theil_u(obs.T)]},
    index=["G (razón de verosimilitud)", "Gamma de Goodman-Kruskal", "Tau-b de Kendall", "Rho de Spearman",
           "Lambda: predecir TALLA conociendo T_GES", "Lambda: predecir T_GES conociendo TALLA",
           "Theil U: TALLA | T_GES", "Theil U: T_GES | TALLA"])
print(f"Prueba G: G = {G:,.1f}, gl = {gl}, p = {pG:.2e}")
otros.round(4)

✏️ **Interpretación:** la **prueba G** (G = 64.580,9; p ≈ 0) coincide con el chi cuadrado en rechazar la independencia. Ambos estadísticos son asintóticamente equivalentes bajo H₀ pero aquí difieren en magnitud porque el χ² está dominado por una sola celda («< 28 semanas × < 40 cm», con esperada de solo 18,4), que aporta ≈ 147.000 de sus 194.519 (≈ 75 %); G pondera menos esas celdas. Las medidas **ordinales** son positivas: gamma = 0,649, tau‑b = 0,329 y rho de Spearman = 0,331. La gamma es mayor porque ignora los pares empatados (muchísimos con solo 3 y 4 categorías), mientras que tau‑b y rho los consideran; por eso sus valores son más bajos y no son comparables con V. El signo positivo confirma que **a más semanas de gestación, mayor talla**, y que la asociación es en gran parte monótona. Las medidas **asimétricas** matizan «qué tan dependiente es una variable de la otra»: conocer el tiempo de gestación reduce el error al predecir la talla en un 26,3 % (lambda), pero conocer la talla casi no mejora la predicción de la gestación (lambda = 0,024), porque «38–41» es la categoría dominante en casi todas las tallas. Con el coeficiente de incertidumbre de Theil, que usa toda la distribución y no solo la moda, ambas direcciones reducen la entropía en una cantidad parecida (U = 0,098 para la talla dada la gestación y 0,118 para la gestación dada la talla, ≈ 10–12 %). **Conclusión metodológica:** que la asociación sea «grande» según V no significa que una variable prediga bien a la otra: dentro de cada grupo de gestación queda mucha incertidumbre sobre la talla (por ejemplo, entre 38 y 41 semanas, el 39,2 % mide 40–49 cm y el 60,8 % mide ≥ 50 cm). Y las medidas asimétricas no determinan quién causa a quién.

### 7.8 Análisis de sensibilidad
Se comprueba si la conclusión depende de las decisiones de agrupación y de la fuente del certificado, recalculando la V de Cramér en tres escenarios alternativos:

1. **Agrupación CIE:** se reemplaza `T_GES` por `T_GES_AGRU_CIE` (del propio DANE), que separa 28–36 de 37–41 semanas, alineado con el corte de 37 semanas de la OMS.
2. **Solo nacimientos de 28 a 41 semanas** (se excluyen los extremadamente prematuros y los de 42 semanas o más), para ver si la asociación se mantiene sin los grupos de cola.
3. **Otra agrupación de gestación:** `T_GES` en solo 3 categorías (< 38, 38–41 y ≥ 42 semanas), que reúne los nacimientos pretérmino y los de 37 semanas.

In [ ]:
def v_cramer(tab):
    tab = np.asarray(tab); nn = tab.sum(); ch = stats.chi2_contingency(tab, correction=False)[0]
    return np.sqrt(ch / (nn * (min(tab.shape) - 1))), ch, (stats.chi2_contingency(tab, correction=False)[3] < 5).mean()

escenarios = {}
# Base
escenarios["Base (agrupación del análisis)"] = ct.to_numpy()

# 1) Agrupación CIE para T_GES
AGR_CIE = {1: "<28", 2: "<28", 3: "28–36", 4: "37–41", 5: "≥42"}
base = df[~df["T_GES_AGRU_CIE"].isin([6, 9]) & talla_ok].copy()
g_cie = pd.Categorical(base["T_GES_AGRU_CIE"].map(AGR_CIE), categories=["<28", "28–36", "37–41", "≥42"], ordered=True)
t_g = pd.Categorical(base["TALLA"].astype(object).map(AGR_TALLA), categories=ORD_TALLA_G, ordered=True)
escenarios["Agrupación CIE de T_GES"] = pd.crosstab(g_cie, t_g).to_numpy()

# 2) Solo 28 a 41 semanas (tabla 2x3)
sub = comp[comp["GES_G"].isin(["28–37", "38–41"])]
escenarios["Solo 28 a 41 semanas"] = pd.crosstab(sub["GES_G"], sub["TALLA_G"]).reindex(index=["28–37", "38–41"], columns=ORD_TALLA_G, fill_value=0).to_numpy()

# 3) T_GES en 3 categorías: <38, 38-41, >=42
g3 = comp["GES_G"].astype(object).map({"<28": "<38", "28–37": "<38", "38–41": "38–41", "≥42": "≥42"})
tab3 = pd.crosstab(g3, comp["TALLA_G"]).reindex(index=["<38", "38–41", "≥42"], columns=ORD_TALLA_G, fill_value=0)
escenarios["T_GES en 3 categorías (<38, 38–41, ≥42)"] = tab3.to_numpy()

filas = []
for nombre, tab in escenarios.items():
    v, ch, frac = v_cramer(tab)
    filas.append({"escenario": nombre, "n": int(tab.sum()), "tabla": f"{tab.shape[0]}x{tab.shape[1]}", "chi2": round(ch), "V de Cramér": round(v, 3), "% celdas esperada<5": round(100 * frac, 1)})
pd.DataFrame(filas).set_index("escenario")

✏️ **Interpretación:** la V de Cramér es **estable (0,466 con la agrupación de la CIE frente a 0,469 con la del análisis)**, así que la conclusión no depende de dónde se corte entre 37 y 38 semanas. Si se excluyen los extremadamente prematuros y los de 42 semanas o más (solo 28 a 41 semanas), V baja a **0,330** (χ² = 47.907; tabla 2 × 3, con df* = 1 los umbrales de Cohen son 0,1; 0,3 y 0,5, así que es un efecto **mediano**): la asociación persiste, pero es menos fuerte, porque buena parte del χ² de la tabla completa viene del grupo de < 28 semanas, cuyo 96,4 % mide menos de 40 cm. Al reunir todas las gestaciones de menos de 38 semanas en una sola categoría, V = 0,242 (χ² = 52.085; 3 × 3, efecto mediano con df* = 2), por el mismo motivo y porque V depende de la forma de la tabla. **Conclusión:** en todos los escenarios la asociación es estadísticamente significativa y de tamaño mediano a grande; incluso entre los nacidos de 28 a 41 semanas, que son casi todos los datos, es mediana (V = 0,33). La magnitud exacta de V depende de cómo se agrupan las categorías, por lo que se reporta junto con los escenarios.

## 8. Respuestas a las preguntas

**1. ¿Cómo se comporta `T_GES`?** Es una variable categórica ordinal muy concentrada: el 72,17 % de los nacimientos tiene 38 a 41 semanas (moda y categoría mediana) y el 27,36 % tiene 28 a 37; las demás categorías suman 0,47 %. La distribución es asimétrica hacia las gestaciones cortas; con la clasificación de la CIE, el 11,1 % de los nacimientos son pretérmino (< 37 semanas) y el 0,39 % extremadamente prematuros.

**2. ¿Cómo se comporta `TALLA_NAC`?** También es categórica ordinal y muy concentrada: el 50,99 % mide 50 a 59 cm (moda y categoría mediana) y el 47,94 % mide 40 a 49 cm; solo el 1,07 % mide menos de 40 cm. La talla mediana está cerca de 50 cm, el límite entre las dos categorías principales, lo que es coherente con los estándares internacionales para recién nacidos a término.

**3. ¿Alguna variable depende de la otra, o están asociadas? ¿Qué tan fuerte?** Están **asociadas**. La prueba chi cuadrado rechaza la independencia (χ² = 194.518,9; 6 gl; p ≈ 0) y las medidas basadas en el estadístico indican una asociación **fuerte**: V de Cramér = 0,47 (efecto grande según Cohen), φ = 0,66, C corregido = 0,68 y T = 0,42. El sentido es positivo: la probabilidad de medir 50 cm o más pasa de 0 % (< 28 semanas) a 25,9 % (28–37), 60,8 % (38–41) y 74,7 % (≥ 42). Por contexto biológico se puede hablar de **dependencia de la talla respecto del tiempo de gestación** y no al revés, porque los estándares INTERGROWTH‑21st muestran que la longitud mediana aumenta con la edad gestacional; el chi cuadrado, en cambio, solo prueba asociación y es simétrico. Las medidas asimétricas (lambda = 0,26; U de Theil ≈ 0,10) indican que conocer la gestación ayuda a predecir la talla, pero deja una incertidumbre considerable. La fuerza depende de la agrupación: V = 0,47 con todas las categorías y 0,33 si solo se consideran los nacidos de 28 a 41 semanas.

## 9. Conclusiones (desde el contexto)

- Casi todos los nacimientos de 2024 ocurren entre las 28 y las 41 semanas (99,9 %) y miden entre 40 y 59 cm (98,9 %). El 11,1 % son pretérmino, una proporción del mismo orden que la estimación mundial de la OMS (más de 1 de cada 10).
- El tiempo de gestación y la talla al nacer están fuertemente asociados (V = 0,47): los recién nacidos más prematuros son, casi siempre, más pequeños. Es lo esperable por fisiología fetal y coincide con los estándares de crecimiento INTERGROWTH‑21st; el resultado también respalda la calidad básica de la información del certificado.
- La asociación se sostiene en las pruebas de sensibilidad (V = 0,47 con la agrupación de la CIE; 0,33 limitada a 28–41 semanas; 0,24 con tres categorías de gestación), aunque su magnitud depende del agrupamiento y está influida por el pequeño grupo de extremadamente prematuros.
- Limitaciones: (i) ambas variables vienen **agrupadas** en intervalos amplios (10 cm; entre 4 y 10 semanas), y la categoría «28 a 37» mezcla prematuros con nacidos en la semana 37; (ii) los faltantes (2,4 %) se concentran en los formularios tipo 2 y 3, de modo que los resultados describen los certificados RUAF‑ND; (iii) la asociación **no se controló** por sexo del recién nacido ni por embarazo múltiple (`MUL_PARTO`), que influyen tanto en la gestación como en la talla; (iv) con n ≈ 443.000 cualquier prueba de hipótesis rechaza, por lo que la interpretación debe apoyarse en las medidas de tamaño del efecto.

## 10. Referencias
- Agresti, A. (2013). *Categorical data analysis* (3.ª ed.). Wiley.
- Bergsma, W. (2013). A bias-correction for Cramér's V and Tschuprow's T. *Journal of the Korean Statistical Society, 42*(3), 323–328.
- Cochran, W. G. (1954). Some methods for strengthening the common χ² tests. *Biometrics, 10*(4), 417–451.
- Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2.ª ed.). Lawrence Erlbaum Associates.
- Cramér, H. (1946). *Mathematical methods of statistics*. Princeton University Press.
- DANE. (2026a). *Estadísticas Vitales – EEVV – 2024* [Ficha técnica, Catálogo Central de Datos, COL‑DANE‑EEVV‑2024]. https://microdatos.dane.gov.co/index.php/catalog/878
- DANE. (2026b). *Diccionario de datos: BD‑EEVV‑Nacimientos‑2024* (variables T_GES, T_GES_AGRU_CIE, TALLA_NAC y TIPOFORMULARIO). https://microdatos.dane.gov.co/index.php/catalog/878/data-dictionary/F46?file_name=BD-EEVV-Nacimientos-2024
- DANE. (2025, 25 de septiembre). *Estadísticas Vitales (EEVV). Nacimientos y defunciones no fetales: cifras definitivas año 2024 y año corrido 2025pr (enero–julio)* [Boletín técnico]. https://www.dane.gov.co/files/operaciones/EEVV/2025/25-sep-2025/bol-EEVV-Isem2025pr.pdf
- Goodman, L. A., & Kruskal, W. H. (1954). Measures of association for cross classifications. *Journal of the American Statistical Association, 49*(268), 732–764.
- Organización Mundial de la Salud. (2023, 10 de mayo). *Preterm birth* [Hoja informativa]. https://www.who.int/en/news-room/fact-sheets/detail/preterm-birth
- Pearson, K. (1900). On the criterion that a given system of deviations from the probable in the case of a correlated system of variables is such that it can be reasonably supposed to have arisen from random sampling. *Philosophical Magazine, 50*(302), 157–175.
- Theil, H. (1970). On the estimation of relationships involving qualitative variables. *American Journal of Sociology, 76*(1), 103–154.
- Villar, J., Cheikh Ismail, L., Victora, C. G., et al. (2014). International standards for newborn weight, length, and head circumference by gestational age and sex: the Newborn Cross-Sectional Study of the INTERGROWTH‑21st Project. *The Lancet, 384*(9946), 857–868. https://doi.org/10.1016/S0140-6736(14)60932-6

## Anexo: código
El código completo está en las celdas anteriores de este cuaderno. Para el informe en PDF, ejecutar todo y usar *Archivo → Imprimir → Guardar como PDF*.